In [ ]:
import os
import sys
import re
import shutil
import sqlite3
import subprocess
import difflib
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ── 项目根目录解析 ──
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from src.utils.cwe import CWEUtils
from src.utils.cve import CVEUtils
from src.utils.formatter import auto_format_c_code_batch, ClangFormatStyle

# ── 超参数 ──
MIN_CODE_LEN       = 50       # 最短代码字符数（过滤过短的无意义代码）
MAX_CODE_LEN       = 65535    # 最长代码字符数（Milvus VARCHAR 上限）
MERGE_FUNCS_AS_CVE = True     # 是否按 (cve_id, commit_id) 合并同源多函数

# ── 文件路径 ──
# 49GB SQL dump 与导入后的 SQLite 数据库都位于 notebook 所在目录的 dataset/ 下
SQL_DUMP_PATH  = Path("dataset/CVEfixes_v1.0.8.sql")
SQLITE_DB_PATH = Path("dataset/CVEfixes.db")
OUTPUT_PATH    = PROJECT_ROOT / (
    "data/raw/vul/CVEfixes_Merged.jsonl" if MERGE_FUNCS_AS_CVE
    else "data/raw/vul/CVEfixes.jsonl"
)

print(f"SQL dump:       {SQL_DUMP_PATH}")
print(f"SQLite DB:      {SQLITE_DB_PATH}")
print(f"Output path:    {OUTPUT_PATH}")
print(f"Merge funcs:    {MERGE_FUNCS_AS_CVE}")
print(f"Code length:    [{MIN_CODE_LEN}, {MAX_CODE_LEN}]")

In [ ]:
# ── SQL dump → SQLite 一次性导入 ──
# CVEfixes 官方发布的是 ~49GB 的 SQL dump，需先导入为 .db 才能进行 SQL 查询。
# 导入耗时 1~3 小时，产物约 30~50GB。第二次运行会自动跳过。

_MIN_DB_SIZE_BYTES = 1 * 1024 * 1024 * 1024  # 1 GB 阈值：低于此判定为导入未完成

def _human_bytes(n: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"


if SQLITE_DB_PATH.exists() and SQLITE_DB_PATH.stat().st_size >= _MIN_DB_SIZE_BYTES:
    print(
        f"检测到已有 .db (大小 {_human_bytes(SQLITE_DB_PATH.stat().st_size)})，跳过导入。"
    )
else:
    if SQLITE_DB_PATH.exists():
        print(
            f"已有 .db 大小 {_human_bytes(SQLITE_DB_PATH.stat().st_size)} 小于阈值 "
            f"{_human_bytes(_MIN_DB_SIZE_BYTES)}，认为导入未完成，将重新导入。"
        )
        SQLITE_DB_PATH.unlink()

    if not SQL_DUMP_PATH.exists():
        raise FileNotFoundError(
            f"未找到 SQL dump 文件: {SQL_DUMP_PATH.resolve()}\n"
            f"请将 CVEfixes_v1.0.8.sql 放置到该路径下。"
        )

    sqlite3_cli = shutil.which("sqlite3")
    if sqlite3_cli is None:
        raise FileNotFoundError(
            "未找到 sqlite3 CLI，无法导入大型 SQL dump。\n"
            "请先安装 sqlite3 (Ubuntu/WSL: sudo apt install sqlite3)，\n"
            "或手动执行: sqlite3 dataset/CVEfixes.db < dataset/CVEfixes_v1.0.8.sql"
        )

    SQLITE_DB_PATH.parent.mkdir(parents=True, exist_ok=True)

    sql_size = SQL_DUMP_PATH.stat().st_size
    print(
        f"开始导入 SQL → SQLite ...\n"
        f"  源文件:   {SQL_DUMP_PATH}  ({_human_bytes(sql_size)})\n"
        f"  目标 DB: {SQLITE_DB_PATH}\n"
        f"  CLI:      {sqlite3_cli}\n"
        f"  注意:     这一步可能耗时 1~3 小时，请耐心等待。"
    )

    with open(SQL_DUMP_PATH, "rb") as f_sql:
        proc = subprocess.Popen(
            [sqlite3_cli, str(SQLITE_DB_PATH)],
            stdin=f_sql,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
            text=True,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line.rstrip())
        ret = proc.wait()

    if ret != 0:
        raise RuntimeError(f"sqlite3 导入失败，退出码 {ret}")

    print(f"\n导入完成，DB 大小: {_human_bytes(SQLITE_DB_PATH.stat().st_size)}")

# ── 验证 ──
with sqlite3.connect(SQLITE_DB_PATH) as _conn:
    quick = _conn.execute("PRAGMA quick_check;").fetchone()
    print(f"PRAGMA quick_check: {quick[0] if quick else 'no result'}")
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
        _conn,
    )
    print(f"已发现 {len(tables)} 张表:")
    for t in tables['name'].tolist():
        cnt = _conn.execute(f"SELECT COUNT(*) FROM '{t}'").fetchone()[0]
        print(f"  {t:<30}: {cnt:>10} 行")

In [ ]:
# ── 关联 SQL 查询：method_change ↔ file_change ↔ commits ↔ fixes ↔ cve + 聚合 cwe ──
# 仅保留 C / C++ 程序语言、code 非空的方法级修改。
# cwe_classification 中一个 CVE 可能对应多个 CWE，使用 GROUP_CONCAT 聚合后在 Python 端解析。

QUERY = """
SELECT
    m.method_change_id,
    cv.cve_id,
    cv.description       AS cve_desc_raw,
    cwe_agg.cwe_ids      AS cwe_ids_raw,
    c.hash               AS commit_id,
    f.file_change_id,
    f.programming_language,
    m.name               AS method_name,
    m.signature,
    m.code,
    m.before_change
FROM method_change m
JOIN file_change f ON f.file_change_id = m.file_change_id
JOIN commits     c ON c.hash           = f.hash
JOIN fixes      fx ON fx.hash          = c.hash
JOIN cve        cv ON cv.cve_id        = fx.cve_id
LEFT JOIN (
    SELECT cve_id, GROUP_CONCAT(cwe_id, ',') AS cwe_ids
    FROM cwe_classification
    GROUP BY cve_id
) cwe_agg ON cwe_agg.cve_id = cv.cve_id
WHERE f.programming_language IN ('C', 'C++')
  AND m.code IS NOT NULL
  AND TRIM(m.code) <> ''
"""

print("执行关联查询（这可能需要数分钟）...")
with sqlite3.connect(SQLITE_DB_PATH) as conn:
    df_raw = pd.read_sql_query(QUERY, conn)

print(f"原始方法级条目: {len(df_raw):>10}")
print(f"内存占用:        {df_raw.memory_usage(deep=True).sum() / 1024**2:>10.2f} MB")

# ── before_change 归一化为 bool ──
# SQLite 中可能以 TEXT 'True'/'False'、INTEGER 0/1 或原生布尔存储
def _to_bool(val):
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, np.integer)):
        return bool(val)
    if isinstance(val, str):
        s = val.strip().lower()
        if s in ('true', '1', 't', 'yes'):
            return True
        if s in ('false', '0', 'f', 'no'):
            return False
    return None

df_raw['before_change'] = df_raw['before_change'].apply(_to_bool)
n_bad_bool = df_raw['before_change'].isna().sum()
if n_bad_bool:
    print(f"丢弃 before_change 解析失败的行: {n_bad_bool}")
    df_raw = df_raw[df_raw['before_change'].notna()].reset_index(drop=True)

# ── 从 cwe_ids_raw (逗号分隔) 解析首个有效 CWE（数值升序后取第一个）──
_CWE_RE = re.compile(r"CWE-(\d+)", re.IGNORECASE)

def _pick_first_cwe(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    text = str(raw)
    matches = _CWE_RE.findall(text)
    if not matches:
        return None
    nums = sorted({int(m) for m in matches})
    return f"CWE-{nums[0]}"

df_raw['cwe_id'] = df_raw['cwe_ids_raw'].apply(_pick_first_cwe)

# 仅保留 cwe_id / cve_id 非空的样本（步骤 2.5）
before = len(df_raw)
df_raw = df_raw[df_raw['cwe_id'].notna() & df_raw['cve_id'].notna()].reset_index(drop=True)
print(f"CWE/CVE 非空过滤: 移除 {before - len(df_raw)} 条，剩余 {len(df_raw)} 条")

df_raw[['cve_id', 'cwe_id', 'commit_id', 'programming_language',
        'method_name', 'before_change']].head()

In [ ]:
# ── before/after 配对 ──
# CVEfixes 中 method_change 表的每个方法对应 2 条记录：before_change=True 为漏洞版本，False 为修复版本。
# 按 (file_change_id, method_name, signature) 分组配对，缺任一侧的方法直接跳过。

PAIR_KEYS = ['file_change_id', 'method_name', 'signature']

records = []
n_missing_pair = 0
for _, grp in tqdm(df_raw.groupby(PAIR_KEYS, sort=False),
                   desc="配对 before/after", total=df_raw[PAIR_KEYS].drop_duplicates().shape[0]):
    vul_rows   = grp[grp['before_change'] == True]
    fixed_rows = grp[grp['before_change'] == False]
    if vul_rows.empty or fixed_rows.empty:
        n_missing_pair += 1
        continue
    v = vul_rows.iloc[0]
    fx = fixed_rows.iloc[0]
    records.append({
        'cve_id':       v['cve_id'],
        'commit_id':    v['commit_id'],
        'cwe_id':       v['cwe_id'],
        'cve_desc_raw': v['cve_desc_raw'],
        'language':     v['programming_language'],
        'vul_code':     v['code'],
        'fixed_code':   fx['code'],
    })

df = pd.DataFrame(records)
print(f"配对成功: {len(df):>8} 条")
print(f"丢弃单边: {n_missing_pair:>8} 条 (缺失 before 或 after)")

# 配对完成后释放原始 DataFrame，腾出内存
del df_raw, records
df.head()

In [ ]:
def _is_valid_str(val):
    """判断值是否为非空、非 None 的字符串。"""
    if val is None:
        return False
    if not isinstance(val, str):
        return False
    s = val.strip()
    return len(s) > 0 and s.lower() != 'none'


# ── 步骤 1：只保留 C 或 C++ 语言（SQL 已过滤，此处再做一次保护）──
df = df[df['language'].isin(['C', 'C++'])]
print(f"步骤1 (C/C++ 语言):        {len(df):>7} 条")

# ── 步骤 2：vul_code 和 fixed_code 非空 ──
df = df[
    df['vul_code'].apply(_is_valid_str)
    & df['fixed_code'].apply(_is_valid_str)
]
print(f"步骤2 (代码非空):          {len(df):>7} 条")

# ── 步骤 3：vul_code != fixed_code（漏洞修复真实有效） ──
df = df[df['vul_code'] != df['fixed_code']]
print(f"步骤3 (vul!=fixed):        {len(df):>7} 条")

# ── 步骤 4：cwe_id 和 cve_id 非空 ──
df = df[
    df['cwe_id'].apply(_is_valid_str)
    & df['cve_id'].apply(_is_valid_str)
]
print(f"步骤4 (CWE/CVE非空):       {len(df):>7} 条")

# ── 步骤 5：代码长度过滤 ──
vul_len   = df['vul_code'].str.len()
fixed_len = df['fixed_code'].str.len()
df = df[
    vul_len.between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')
    & fixed_len.between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')
]
print(f"步骤5 (代码长度合法):      {len(df):>7} 条")

df = df.reset_index(drop=True)
print(f"\n清洗后总样本数: {len(df)}")
df[['cwe_id', 'cve_id', 'language', 'vul_code', 'fixed_code']].head()

In [ ]:
SEP = "\n\n/* ---- next function ---- */\n\n"

if MERGE_FUNCS_AS_CVE:
    print("启用同源 CVE 合并 (MERGE_FUNCS_AS_CVE=True)...")

    # 注意：直接使用 groupby(...).apply(...) 在新版 pandas 中存在分组键与返回值列名冲突
    # 的问题（cve_id/commit_id 会进入 MultiIndex，与 Series 同名键冲突后被静默丢弃，
    # 导致 cve_id 列在结果中消失）。这里改为手动遍历分组并显式构造列表，行为更可控。
    merged_rows = []
    for (cve_id, commit_id), grp in df.groupby(['cve_id', 'commit_id'], sort=False):
        first = grp.iloc[0].to_dict()
        first['cve_id']     = cve_id
        first['commit_id']  = commit_id
        first['vul_code']   = SEP.join(grp['vul_code'].tolist())
        first['fixed_code'] = SEP.join(grp['fixed_code'].tolist())
        merged_rows.append(first)

    df = pd.DataFrame(merged_rows).reset_index(drop=True)

    # 合并后再次过滤超长代码
    df = df[
        df['vul_code'].str.len().between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')
        & df['fixed_code'].str.len().between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')
    ].reset_index(drop=True)

    print(f"合并后样本数: {len(df)}")
else:
    print(f"跳过同源 CVE 合并 (MERGE_FUNCS_AS_CVE=False)，当前样本数: {len(df)}")

df[['cve_id', 'cwe_id', 'language', 'vul_code', 'fixed_code']].head()

In [ ]:
print("格式化 vul_code ...")
vul_results = auto_format_c_code_batch(
    df['vul_code'].tolist(),
    style=ClangFormatStyle.LLVM,
    on_error="keep",
    show_progress=True,
)
df['vul_code']      = [r[0] for r in vul_results]
df['_lang_vul']     = [r[1].language for r in vul_results]   # 'c' / 'cpp'

print("格式化 fixed_code ...")
fixed_results = auto_format_c_code_batch(
    df['fixed_code'].tolist(),
    style=ClangFormatStyle.LLVM,
    on_error="keep",
    show_progress=True,
)
df['fixed_code']    = [r[0] for r in fixed_results]
df['_lang_fixed']   = [r[1].language for r in fixed_results]

# 统计格式化置信度情况
low_conf_vul   = sum(1 for r in vul_results   if r[1].confidence < 0.6)
low_conf_fixed = sum(1 for r in fixed_results if r[1].confidence < 0.6)
print(f"\nvul_code   低置信度样本数 (<0.6): {low_conf_vul}")
print(f"fixed_code 低置信度样本数 (<0.6): {low_conf_fixed}")
print(f"格式化完成，总样本数: {len(df)}")

# ── 步骤3-bis：格式化后再次过滤 vul_code == fixed_code ──
# clang-format 会将仅存在空白符/缩进差异的代码对统一为相同字符串，
# 因此必须在格式化后重新执行此过滤，才能保证 diff 非空。
before_dedup = len(df)
df = df[df['vul_code'] != df['fixed_code']].reset_index(drop=True)
removed = before_dedup - len(df)
print(f"\n步骤3-bis (格式化后 vul!=fixed): 移除 {removed} 条，剩余 {len(df)} 条")

In [ ]:
# ── 用代码内容自动检测的语言重写 language 列 ──
# - 任一侧（vul_code / fixed_code）检测为 cpp → 整条样本判定为 Cpp
# - 双侧均为 c                          → 整条样本判定为 C
# - 命名约定：'C' / 'Cpp'（小写 cpp 统一改写为 'Cpp'，符合项目命名规范）

def _resolve_language(lv: str, lf: str) -> str:
    return "Cpp" if ("cpp" in (lv, lf)) else "C"

df['language'] = [
    _resolve_language(lv, lf)
    for lv, lf in zip(df['_lang_vul'], df['_lang_fixed'])
]
df = df.drop(columns=['_lang_vul', '_lang_fixed'])

print(f"language 分布:\n{df['language'].value_counts()}")
df[['cve_id', 'cwe_id', 'language']].head()

In [ ]:
cwe_utils = CWEUtils(PROJECT_ROOT / "data/raw/cwe/cwec_v4.19.1.xml")

def get_cwe_desc(cwe_field):
    """将 cwe_id 字段转换为 CWEUtils prompt 文本。

    支持字符串 / 列表 / numpy 数组格式；找不到时返回 None。
    """
    if cwe_field is None:
        return None
    cwe_id = None
    if isinstance(cwe_field, np.ndarray) and len(cwe_field) > 0:
        cwe_id = str(cwe_field[0])
    elif isinstance(cwe_field, list) and len(cwe_field) > 0:
        cwe_id = str(cwe_field[0])
    elif isinstance(cwe_field, str):
        m = re.search(r"CWE-\d+", cwe_field, re.IGNORECASE)
        cwe_id = m.group() if m else cwe_field.strip().strip("[]'\"").split(",")[0].strip().strip("'\"")
    if not cwe_id or not cwe_id.strip():
        return None
    prompt = cwe_utils.get_prompt(cwe_id.strip())
    if "not found in the CWE dataset" in prompt:
        return None
    return prompt

df['cwe_desc'] = df['cwe_id'].apply(get_cwe_desc)

filled = df['cwe_desc'].notna().sum()
print(f"总样本数:              {len(df)}")
print(f"cwe_desc 已填充:       {filled}")
print(f"cwe_desc 为空:         {len(df) - filled}")
print(f"cwe_desc 填充率:       {filled / len(df) * 100:.2f}%")
df[['cwe_id', 'cwe_desc']].head()

In [ ]:
cve_utils = CVEUtils(PROJECT_ROOT / "data/raw/cwe/CVE_v2026.03.18.json")

def _normalize_cve_desc_raw(val):
    """将 cve.description 字段归一化为字符串，过滤空值。"""
    if val is None:
        return None
    if isinstance(val, str):
        s = val.strip()
        return s if s and s.lower() != 'none' else None
    return None

def get_cve_desc(row):
    """CVE 描述获取策略：优先使用 CVEfixes 原始 description 字段，不足时用 CVEUtils 补充。"""
    # 1. 优先使用 CVEfixes 自带的 cve.description
    raw_desc = _normalize_cve_desc_raw(row.get('cve_desc_raw'))
    if raw_desc:
        return raw_desc

    # 2. 回退：通过 CVEUtils 按 cve_id 查询
    cve_id = row.get('cve_id')
    if not cve_id or not str(cve_id).strip():
        return None
    prompt = cve_utils.get_prompt(str(cve_id).strip())
    if "was not found in the CVE dataset" in prompt:
        return None
    return prompt

df['cve_desc'] = df.apply(get_cve_desc, axis=1)

filled   = df['cve_desc'].notna().sum()
from_raw = df['cve_desc_raw'].apply(_normalize_cve_desc_raw).notna().sum()
print(f"总样本数:                       {len(df)}")
print(f"cve_desc 已填充:                {filled}")
print(f"  └─ 来自 cve.description:      {from_raw}")
print(f"  └─ 来自 CVEUtils 补充:        {filled - from_raw}")
print(f"cve_desc 为空:                  {len(df) - filled}")
print(f"cve_desc 填充率:                {filled / len(df) * 100:.2f}%")
df[['cve_id', 'cve_desc']].head()

In [ ]:
def compute_diff(vul_code: str, fixed_code: str) -> str:
    """使用 difflib.unified_diff 计算函数级代码变更差异。"""
    a = vul_code.splitlines(keepends=True)
    b = fixed_code.splitlines(keepends=True)
    diff_lines = list(difflib.unified_diff(a, b, fromfile="vul_code", tofile="fixed_code"))
    return "".join(diff_lines) if diff_lines else ""

df['diff'] = df.apply(lambda r: compute_diff(r['vul_code'], r['fixed_code']), axis=1)

empty_diff = (df['diff'] == '').sum()
print(f"总样本数:         {len(df)}")
print(f"diff 非空:        {len(df) - empty_diff}")
print(f"diff 为空:        {empty_diff}  （vul_code 与 fixed_code 完全一致，理论上已在步骤3过滤）")
df[['vul_code', 'fixed_code', 'diff']].head(1)

In [ ]:
df = df.reset_index(drop=True)
df['id']     = df.index        # 0-based 自增整数主键
df['source'] = 'CVEfixes'

print(f"id 范围:   0 ~ {df['id'].max()}")
print(f"source:    {df['source'].unique().tolist()}")
print(f"总样本数:  {len(df)}")
df[['id', 'source', 'cwe_id', 'cve_id', 'language']].head()

In [ ]:
FINAL_COLS = [
    'id', 'source', 'language',
    'cwe_id', 'cve_id',
    'cwe_desc', 'cve_desc',
    'vul_code', 'fixed_code',
    'diff',
]

df_out = df[FINAL_COLS].copy()

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_out.to_json(
    OUTPUT_PATH,
    orient='records',
    lines=True,
    force_ascii=False,
)

print(f"导出完成: {OUTPUT_PATH}")
print(f"总样本数: {len(df_out)}")
print(f"\n字段统计：")
for col in FINAL_COLS:
    null_count = df_out[col].isna().sum()
    fill_rate  = (1 - null_count / len(df_out)) * 100
    print(f"  {col:<14}: 填充率 {fill_rate:6.2f}%  (空值 {null_count} 条)")
df_out.head()

In [ ]:
# end of preprocessing pipeline